# Limitation 3: Making the Reduced Force Law Explicit

## Brief recap of the current model state

The repository already contains the minimal 2D event-based Monte Carlo model, the robustness sweeps, the graded shielding alternative, and the recent endogenous `body_surface` source model. The current scientific picture before this notebook iteration is:

- imposed external fields can produce attraction at some gaps under shielding assumptions
- the reduced body-generated source field produces strong repulsion across the tested gap range
- this means the source-process assumption is decisive, not secondary

This notebook does **not** replace that source-process result. It addresses the next limitation: the interaction was being inferred from side-impulse bookkeeping rather than from an explicit reduced force law.

## Why the force-law limitation matters

The bookkeeping model logs hits and cumulative impulse on the inner and outer long sides, then infers a gap-closing force from their difference. That is transparent, but the mapping from event exposure to net force is still indirect.

A reduced alternative should make the force map explicit without pretending to be calibrated hydrodynamics. The goal here is therefore modest: test whether the qualitative attraction/repulsion conclusions survive when the event field is mapped to force by a different, still transparent rule.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_ROOT = PROJECT_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.append(str(SRC_ROOT))

from float_sim.event_model import (
    ModelParameters,
    ShieldingModel,
    SourceField,
    event_count_from_source_density,
    plot_ensemble_metric,
    plot_geometry,
    plot_side_metrics,
    run_ensemble,
    run_gap_ensemble_sweep,
    same_force_sign,
    simulate_batch,
)

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 120
np.set_printoptions(precision=3, suppress=True)


## Reduced alternative: explicit boundary-pressure integration

The new alternative keeps the same sampled event field, attenuation law, shielding rule, and self-emission exclusion rule. Only the **force inference** changes.

Two formulations are compared on the same Monte Carlo event realizations:

1. **Bookkeeping inference**: average inward impulse on the inner and outer long sides, then infer gap-closing force from outer minus inner cumulative impulse.
2. **Explicit boundary-pressure rule**: sample the full rectangle perimeter, compute a reduced local pressure contribution from each event onto each boundary sample, multiply by the local inward normal and boundary weight, then sum those force-density contributions to obtain an explicit net force vector.

This is still a toy model in reduced units. It is not a calibrated hydrodynamic law. Its purpose is only to expose the event-to-force mapping more directly and to test whether the qualitative conclusions are robust to that choice.

In [ ]:
BASE_SEED = 240
ENSEMBLE_REPEATS = 8
BASE_SOURCE_DENSITY = 8.0
GAPS = np.array([0.1, 0.3, 0.6, 1.0, 1.4, 1.8])
REFERENCE_CASES = [
    ('external / graded / gap 0.3', 0.3, SourceField(), ShieldingModel.graded(minimum_transmission=0.15, occlusion_decay_length=0.25)),
    ('external / graded / gap 1.4', 1.4, SourceField(), ShieldingModel.graded(minimum_transmission=0.15, occlusion_decay_length=0.25)),
    ('body / graded / gap 0.3', 0.3, SourceField(model='body_surface', emission_offset=0.12), ShieldingModel.graded(minimum_transmission=0.15, occlusion_decay_length=0.25)),
    ('body / graded / gap 1.0', 1.0, SourceField(model='body_surface', emission_offset=0.12), ShieldingModel.graded(minimum_transmission=0.15, occlusion_decay_length=0.25)),
]

BASE_PARAMS = ModelParameters(
    body_length=3.0,
    body_width=0.4,
    domain_half_length=6.0,
    domain_half_width=4.0,
    attenuation_length=2.5,
    attenuation_power=0.0,
    mean_wave_amplitude=1.0,
    side_samples=17,
    mobility=0.002,
)

GRADED_SHIELDING = ShieldingModel.graded(minimum_transmission=0.15, occlusion_decay_length=0.25)
NO_SHIELDING = ShieldingModel.none()
EXTERNAL_FIELD = SourceField()
BODY_FIELD = SourceField(model='body_surface', emission_offset=0.12)

def events_for(params: ModelParameters, density: float = BASE_SOURCE_DENSITY) -> int:
    return event_count_from_source_density(density, params)

def gap_sweep(source_field: SourceField, shielding_model: ShieldingModel, *, repeats: int = ENSEMBLE_REPEATS):
    return run_gap_ensemble_sweep(
        gaps=GAPS,
        params=BASE_PARAMS,
        n_events=events_for(BASE_PARAMS),
        repeats=repeats,
        seed=BASE_SEED,
        source_field=source_field,
        shielding_model=shielding_model,
    )

def ensemble_at_gap(gap: float, source_field: SourceField, shielding_model: ShieldingModel, *, repeats: int = ENSEMBLE_REPEATS, seed: int = BASE_SEED):
    return run_ensemble(
        gap=gap,
        params=BASE_PARAMS,
        n_events=events_for(BASE_PARAMS),
        repeats=repeats,
        seed=seed,
        source_field=source_field,
        shielding_model=shielding_model,
    )

def summary_rows(summaries):
    rows = []
    for summary in summaries:
        ratio = np.nan if np.isclose(summary.mean_gap_closing_force, 0.0) else summary.explicit_mean_gap_closing_force / summary.mean_gap_closing_force
        rows.append({
            'gap': round(summary.gap, 2),
            'bookkeeping': round(summary.mean_gap_closing_force, 3),
            'explicit': round(summary.explicit_mean_gap_closing_force, 3),
            'explicit_minus_bookkeeping': round(summary.force_law_gap_difference, 3),
            'explicit_to_bookkeeping_ratio': None if np.isnan(ratio) else round(float(ratio), 3),
            'sign_agreement_rate': round(summary.force_law_sign_agreement_rate, 2),
            'bookkeeping_system_net': round(summary.mean_system_net_force_y, 3),
            'explicit_system_net': round(summary.explicit_mean_system_net_force_y, 3),
        })
    return rows

def case_statistics(records):
    bookkeeping = np.array([record.mean_gap_closing_force for record in records], dtype=float)
    explicit = np.array([record.explicit_mean_gap_closing_force for record in records], dtype=float)
    ratio = explicit / bookkeeping
    agreement = [same_force_sign(left, right) for left, right in zip(bookkeeping, explicit, strict=True)]
    return {
        'bookkeeping_mean': float(bookkeeping.mean()),
        'explicit_mean': float(explicit.mean()),
        'ratio_mean': float(np.mean(ratio)),
        'ratio_std': float(np.std(ratio, ddof=0)),
        'corr': float(np.corrcoef(bookkeeping, explicit)[0, 1]),
        'sign_agreement_rate': float(np.mean(agreement)),
        'mean_abs_bookkeeping_system_net_fraction': float(np.mean(np.abs([record.system_net_force_y for record in records])) / np.mean(np.abs(bookkeeping))),
        'mean_abs_explicit_system_net_fraction': float(np.mean(np.abs([record.explicit_system_net_force_y for record in records])) / np.mean(np.abs(explicit))),
    }

def plot_sign_agreement(ax, summaries, *, label: str, color: str):
    gaps = np.array([summary.gap for summary in summaries], dtype=float)
    rates = np.array([summary.force_law_sign_agreement_rate for summary in summaries], dtype=float)
    ax.plot(gaps, rates, marker='o', color=color, label=label)
    ax.set_ylim(-0.02, 1.02)
    ax.set_xlabel('Edge-to-edge gap')
    ax.set_ylabel('Agreement rate across seeds')
    ax.set_title('Force-law sign agreement')
    ax.legend(loc='best')

def plot_force_scatter(ax, cases):
    palette = ['#1d3557', '#457b9d', '#e76f51', '#2a9d8f']
    all_values = []
    for color, (label, records) in zip(palette, cases.items(), strict=True):
        bookkeeping = np.array([record.mean_gap_closing_force for record in records], dtype=float)
        explicit = np.array([record.explicit_mean_gap_closing_force for record in records], dtype=float)
        all_values.extend(bookkeeping.tolist())
        all_values.extend(explicit.tolist())
        ax.scatter(bookkeeping, explicit, color=color, alpha=0.8, s=28, label=label)
    limit = max(abs(value) for value in all_values)
    ax.plot([-limit, limit], [-limit, limit], linestyle='--', color='0.5', linewidth=1.0)
    ax.set_xlabel('Bookkeeping gap-closing force')
    ax.set_ylabel('Explicit gap-closing force')
    ax.set_title('Per-run force comparison')
    ax.legend(loc='best', fontsize=8)

def plot_ratio_boxplot(ax, cases):
    labels = list(cases)
    values = []
    for label in labels:
        records = cases[label]
        bookkeeping = np.array([record.mean_gap_closing_force for record in records], dtype=float)
        explicit = np.array([record.explicit_mean_gap_closing_force for record in records], dtype=float)
        values.append(explicit / bookkeeping)
    ax.boxplot(values, tick_labels=labels, showmeans=True)
    ax.axhline(1.0, color='0.4', linestyle='--', linewidth=1.0)
    ax.set_ylabel('Explicit / bookkeeping')
    ax.set_title('Per-run force ratio')
    ax.tick_params(axis='x', rotation=15)


## Ensemble methodology

The event field, attenuation law, shielding rule, gap grid, source density, and deterministic seed schedule are held fixed. Each point below is an ensemble over multiple seeds, so the two force formulations are compared on the same reduced model rather than on separately tuned setups.

Two source processes remain in view because the previous iteration showed they matter strongly:

- `uniform` external source field
- `body_surface` endogenous source field with a small outward emission offset

The main robustness question is therefore: does the qualitative attraction/repulsion story survive the force-law change once the source-process choice is held fixed?

In [ ]:
external_batch = simulate_batch(
    gap=0.6,
    params=BASE_PARAMS,
    rng=np.random.default_rng(BASE_SEED),
    n_events=events_for(BASE_PARAMS),
    source_field=EXTERNAL_FIELD,
    shielding_model=GRADED_SHIELDING,
)
body_batch = simulate_batch(
    gap=0.6,
    params=BASE_PARAMS,
    rng=np.random.default_rng(BASE_SEED),
    n_events=events_for(BASE_PARAMS),
    source_field=BODY_FIELD,
    shielding_model=GRADED_SHIELDING,
)

fig, axes = plt.subplots(2, 4, figsize=(18, 8.5))
plot_geometry(axes[0, 0], external_batch, max_events=300)
plot_side_metrics(axes[0, 1], external_batch, metric='hits')
plot_side_metrics(axes[0, 2], external_batch, metric='impulse')
axes[0, 3].bar(['bookkeeping', 'explicit'], [external_batch.mean_gap_closing_force, external_batch.explicit_mean_gap_closing_force], color=['#1d3557', '#457b9d'])
axes[0, 3].axhline(0.0, color='0.4', linestyle='--', linewidth=1.0)
axes[0, 3].set_title('External field: representative force comparison')
axes[0, 3].set_ylabel('Gap-closing force')

plot_geometry(axes[1, 0], body_batch, max_events=300)
plot_side_metrics(axes[1, 1], body_batch, metric='hits')
plot_side_metrics(axes[1, 2], body_batch, metric='impulse')
axes[1, 3].bar(['bookkeeping', 'explicit'], [body_batch.mean_gap_closing_force, body_batch.explicit_mean_gap_closing_force], color=['#e76f51', '#2a9d8f'])
axes[1, 3].axhline(0.0, color='0.4', linestyle='--', linewidth=1.0)
axes[1, 3].set_title('Body-generated field: representative force comparison')
axes[1, 3].set_ylabel('Gap-closing force')

axes[0, 0].set_title('External field: representative batch')
axes[1, 0].set_title('Body-generated field: representative batch')
fig.tight_layout()
plt.show()

print({
    'external_bookkeeping': round(external_batch.mean_gap_closing_force, 3),
    'external_explicit': round(external_batch.explicit_mean_gap_closing_force, 3),
    'body_bookkeeping': round(body_batch.mean_gap_closing_force, 3),
    'body_explicit': round(body_batch.explicit_mean_gap_closing_force, 3),
})


## Results comparing old vs new force inference

The next comparison keeps the source process fixed and asks whether the sign of the interaction changes when the force law changes. The explicit formulation is allowed to differ in magnitude because it integrates a reduced pressure density over the full boundary rather than inferring force only from long-side bookkeeping.

In [ ]:
external_no_shield = gap_sweep(EXTERNAL_FIELD, NO_SHIELDING)
external_graded = gap_sweep(EXTERNAL_FIELD, GRADED_SHIELDING)
body_no_shield = gap_sweep(BODY_FIELD, NO_SHIELDING)
body_graded = gap_sweep(BODY_FIELD, GRADED_SHIELDING)

fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharex=True)

plot_ensemble_metric(axes[0, 0], external_no_shield, metric='gap_closing_force', label='bookkeeping / no shielding', color='#6c757d')
plot_ensemble_metric(axes[0, 0], external_no_shield, metric='explicit_gap_closing_force', label='explicit / no shielding', color='#adb5bd')
plot_ensemble_metric(axes[0, 0], external_graded, metric='gap_closing_force', label='bookkeeping / graded', color='#1d3557')
plot_ensemble_metric(axes[0, 0], external_graded, metric='explicit_gap_closing_force', label='explicit / graded', color='#457b9d')
axes[0, 0].set_title('External source field')

plot_ensemble_metric(axes[0, 1], body_no_shield, metric='gap_closing_force', label='bookkeeping / no shielding', color='#bc6c25')
plot_ensemble_metric(axes[0, 1], body_no_shield, metric='explicit_gap_closing_force', label='explicit / no shielding', color='#dda15e')
plot_ensemble_metric(axes[0, 1], body_graded, metric='gap_closing_force', label='bookkeeping / graded', color='#e76f51')
plot_ensemble_metric(axes[0, 1], body_graded, metric='explicit_gap_closing_force', label='explicit / graded', color='#2a9d8f')
axes[0, 1].set_title('Body-generated source field')

plot_ensemble_metric(axes[1, 0], external_graded, metric='force_law_gap_difference', label='external / graded', color='#457b9d')
plot_ensemble_metric(axes[1, 0], body_graded, metric='force_law_gap_difference', label='body / graded', color='#2a9d8f')
axes[1, 0].set_title('Explicit minus bookkeeping')

plot_sign_agreement(axes[1, 1], external_graded, label='external / graded', color='#457b9d')
plot_sign_agreement(axes[1, 1], body_graded, label='body / graded', color='#2a9d8f')

fig.tight_layout()
plt.show()

print('external / graded')
for row in summary_rows(external_graded):
    print(row)
print()
print('body / graded')
for row in summary_rows(body_graded):
    print(row)


## Diagnostics and controls

Mean curves alone are not enough. The diagnostics below check whether the force-law comparison is being driven by sign flips, unstable seed-by-seed behavior, or loss of the expected symmetry control.

In [ ]:
diagnostic_cases = {
    label: ensemble_at_gap(gap, field, shielding)
    for label, gap, field, shielding in REFERENCE_CASES
}

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

plot_ensemble_metric(axes[0, 0], external_graded, metric='system_net_force', label='bookkeeping / external graded', color='#1d3557')
plot_ensemble_metric(axes[0, 0], external_graded, metric='explicit_system_net_force', label='explicit / external graded', color='#457b9d')
plot_ensemble_metric(axes[0, 0], body_graded, metric='system_net_force', label='bookkeeping / body graded', color='#e76f51')
plot_ensemble_metric(axes[0, 0], body_graded, metric='explicit_system_net_force', label='explicit / body graded', color='#2a9d8f')
axes[0, 0].set_title('Symmetry control: system net force')

plot_ensemble_metric(axes[0, 1], external_graded, metric='mean_abs_net_force', label='bookkeeping / external graded', color='#1d3557')
plot_ensemble_metric(axes[0, 1], external_graded, metric='explicit_mean_abs_net_force', label='explicit / external graded', color='#457b9d')
plot_ensemble_metric(axes[0, 1], body_graded, metric='mean_abs_net_force', label='bookkeeping / body graded', color='#e76f51')
plot_ensemble_metric(axes[0, 1], body_graded, metric='explicit_mean_abs_net_force', label='explicit / body graded', color='#2a9d8f')
axes[0, 1].set_title('Mean absolute body force scale')

plot_force_scatter(axes[1, 0], diagnostic_cases)
plot_ratio_boxplot(axes[1, 1], diagnostic_cases)

fig.tight_layout()
plt.show()

for label, records in diagnostic_cases.items():
    stats = case_statistics(records)
    print(label)
    print({
        'bookkeeping_mean': round(stats['bookkeeping_mean'], 3),
        'explicit_mean': round(stats['explicit_mean'], 3),
        'ratio_mean': round(stats['ratio_mean'], 3),
        'ratio_std': round(stats['ratio_std'], 3),
        'corr': round(stats['corr'], 3),
        'sign_agreement_rate': round(stats['sign_agreement_rate'], 2),
        'bookkeeping_system_net_fraction': round(stats['mean_abs_bookkeeping_system_net_fraction'], 3),
        'explicit_system_net_fraction': round(stats['mean_abs_explicit_system_net_fraction'], 3),
    })
    print()


## Interpretation

Across the tested sweeps and seeds, the qualitative interaction is **robust to this force-law change in sign, but not in magnitude**. In the present reduced model:

- the explicit boundary-pressure rule and the original bookkeeping rule agree on attraction vs repulsion across the tested gaps
- the explicit rule typically produces larger force magnitudes, often by about a factor of three away from the external-field zero crossing
- the body-generated source field still yields robust repulsion across the tested gap range under both formulations
- the externally imposed field with graded shielding still yields attraction at smaller gaps and weakens or changes sign at larger gaps under both formulations

So the main conclusion from the previous iteration survives this one: the **source-process assumption still matters more than this particular force-law choice** in the current reduced model.

Where the two formulations differ most interpretively is near the external-field zero crossing. Around that regime, both forces are small enough that ratios become unstable and system-net-force diagnostics look worse simply because the interaction denominator is close to zero. That is a fragility of the near-null regime, not evidence that one formulation restores attraction or repulsion by itself.

## Limitations

This notebook still does **not** provide a calibrated hydrodynamic force law. The explicit alternative is only a more direct reduced mapping from event field to net force. Important simplifications remain:

- the event field is still point-based rather than a resolved wave field
- the pressure contribution is defined by a reduced directional attenuation rule, not by fluid mechanics
- boundary sampling density and rectangle geometry still matter
- rotation, tangential response, reflection, and interference remain excluded

The correct reading is therefore limited: under this family of reduced assumptions, the sign of the interaction appears robust to the force-law replacement tested here, while the magnitude is formulation-dependent.

## Recommended next step

The next sensible step is **not** to add high-complexity fluid simulation. It is to keep both reduced force formulations and test whether the same qualitative source-process split survives a slightly richer event-field model, such as directional outward emission, angular emission kernels, or a short-range reflected component. That would probe whether the current sign robustness is specific to the force law, or whether it persists when the endogenous source model itself becomes less isotropic.